In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import minmax_scale
from sklearn.model_selection import KFold

import tensorflow as tf

import keras
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Flatten, MaxPool2D, Conv2D
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import L1L2, L1, L2
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

from tqdm import tqdm
import gc

def draw_feature(H, name = 'Heatmap'):
    plt.clf()
    plt.imshow(H.T, origin='lower120000')
    plt.colorbar()
    plt.title(name)
    plt.show()

def get_binding_feature(feature, bin_size):
  a = np.zeros((bin_size*3, bin_size))
  xc1 = int(bin_size/6)
  xc2 = xc1 + int(bin_size/3)
  xc3 = xc2 + int(bin_size/3)
  xc4 = int(bin_size/4)
  xc5 = xc4 + int(bin_size/2)
  xc6 = xc5 + int((bin_size-xc5)/2)

  yc13 = int(bin_size/4)
  yc45 = int(bin_size/4) + int(bin_size/2)

  a[yc13][xc1] = feature[0]
  a[yc13][xc2] = feature[1]
  a[yc13][xc3] = feature[2]
  a[yc45][xc4] = feature[3]
  a[yc45][xc5] = feature[4]
  a[yc45][xc6] = feature[5]

  a[yc13-1][xc1], a[yc13-1][xc1-1],  a[yc13-1][xc1+1], a[yc13][xc1-1], a[yc13][xc1+1], a[yc13+1][xc1], a[yc13+1][xc1-1],  a[yc13+1][xc1+1]= feature[0], feature[0], feature[0], feature[0], feature[0], feature[0], feature[0], feature[0]
  a[yc13-1][xc2], a[yc13-1][xc2-1],  a[yc13-1][xc2+1], a[yc13][xc2-1], a[yc13][xc2+1], a[yc13+1][xc2], a[yc13+1][xc2-1],  a[yc13+1][xc2+1]= feature[1], feature[1], feature[1], feature[1], feature[1], feature[1], feature[1], feature[1]
  a[yc13-1][xc3], a[yc13-1][xc3-1],  a[yc13-1][xc3+1], a[yc13][xc3-1], a[yc13][xc3+1], a[yc13+1][xc3], a[yc13+1][xc3-1],  a[yc13+1][xc3+1]= feature[2], feature[2], feature[2], feature[2], feature[2], feature[2], feature[2], feature[2]
  a[yc45-1][xc4], a[yc45-1][xc4-1],  a[yc45-1][xc4+1], a[yc45][xc4-1], a[yc45][xc4+1], a[yc45+1][xc4], a[yc45+1][xc4-1],  a[yc45+1][xc4+1]= feature[3], feature[3], feature[3], feature[3], feature[3], feature[3], feature[3], feature[3]
  a[yc45-1][xc5], a[yc45-1][xc5-1],  a[yc45-1][xc5+1], a[yc45][xc5-1], a[yc45][xc5+1], a[yc45+1][xc5], a[yc45+1][xc5-1],  a[yc45+1][xc5+1]= feature[4], feature[4], feature[4], feature[4], feature[4], feature[4], feature[4], feature[4]
  a[yc45-1][xc6], a[yc45-1][xc6-1],  a[yc45-1][xc6+1], a[yc45][xc6-1], a[yc45][xc6+1], a[yc45+1][xc6], a[yc45+1][xc6-1],  a[yc45+1][xc6+1]= feature[5], feature[5], feature[5], feature[5], feature[5], feature[5], feature[5], feature[5]
  a = np.array(a)
  
  return a

2024-08-05 02:02:30.125507: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-05 02:02:30.125755: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-05 02:02:30.319055: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
# Datos

# Modelo 

model = tf.keras.models.load_model("./models/CNN_CV_Model.keras")

# lnc
lnc_doc2vec_dict = pd.read_csv("../Data_Demo/embedding/Diccionarios/lnc_doc2vec_dict.txt", header=None, index_col=0)
lnc_ctd_dict = pd.read_csv("../Data_Demo/embedding/Diccionarios/lnc_ctd_dict.txt", header=None, index_col=0)
lnc_role2vec_dict = pd.read_csv("../Data_Demo/embedding/Diccionarios/lnc_role2vec_dict.txt", header=None, index_col=0)
lnc_kmer_dict = pd.read_csv("../Data_Demo/embedding/Diccionarios/lnc_kmer_dict.txt", header=None, index_col=0)
ss_loops_dict = pd.read_csv("../Data_Demo/lncRNA/binding_features_lnc.txt", header=None, index_col=0)

# precursor
mir_doc2vec_dict = pd.read_csv("../Data_Demo/embedding/Diccionarios/mir_doc2vec_dict.txt", header=None, index_col=0)
mir_ctd_dict = pd.read_csv("../Data_Demo/embedding/Diccionarios/mir_ctd_dict.txt", header=None, index_col=0)
mir_role2vec_dict = pd.read_csv("../Data_Demo/embedding/Diccionarios/mir_role2vec_dict.txt", header=None, index_col=0)
mir_kmer_dict = pd.read_csv("../Data_Demo/embedding/Diccionarios/mir_kmer_dict.txt", header=None, index_col=0)
ss_mir_loops_dict = pd.read_csv("../Data_Demo/miRNA/miRNA_Precursor/binding_features_miRNA_precursor.txt", header=None, index_col=0)

# especifico
mir_especifico_doc2vec_dict = pd.read_csv("../Data_Demo/embedding/Diccionarios/mir_especifico_doc2vec_dict.txt", header=None, index_col=0)
mir_especifico_ctd_dict = pd.read_csv("../Data_Demo/embedding/Diccionarios/mir_especifico_ctd_dict.txt", header=None, index_col=0)
mir_especifico_role2vec_dict = pd.read_csv("../Data_Demo/embedding/Diccionarios/mir_especifico_role2vec_dict.txt", header=None, index_col=0)
mir_especifico_kmer_dict = pd.read_csv("../Data_Demo/embedding/Diccionarios/mir_especifico_kmer_dict.txt", header=None, index_col=0)

In [4]:
bin_size=32

with open('../Data_Demo/lnc_mir_list.txt', 'r') as file:
    lines = file.readlines()

rows = []
indices = []

combinations = ss_mir_loops_dict.index.tolist()

index_ss_lops = ss_loops_dict.index

test_lnc = [element.split('_')[0] for element in index_ss_lops]
test_mir = [element.split('_')[1] for element in index_ss_lops]

for i in tqdm(range(len(test_lnc))):
    lnc = test_lnc[i]
    mir = test_mir[i]
    if f'{lnc}_{mir}' in combinations:
        indices.append(f'{lnc}_{mir}')
        lnc_vec = lnc_kmer_dict.loc[lnc].tolist()
        mir_vec = mir_kmer_dict.loc[mir].tolist()
        mir_especifico_vec = mir_especifico_kmer_dict.loc[mir].tolist()
        assert len(lnc_vec) == len(mir_vec) == len(mir_especifico_vec), f'Vector lengths are inconsistent ctd {len(lnc_vec)} {len(mir_vec)} {len(mir_especifico_vec)}'
        H1, xedges, yedges = np.histogram2d(lnc_vec, mir_vec, bins=bin_size, density=True)
        H2, xedges, yedges = np.histogram2d(lnc_vec, mir_especifico_vec, bins=bin_size, density=True)
        H3, xedges, yedges = np.histogram2d(mir_especifico_vec, mir_vec, bins=bin_size, density=True)
        H = np.vstack((H1, H2, H3))
        H = minmax_scale(H)
        H_kmer = H[:, :, np.newaxis]

        lnc_vec = lnc_doc2vec_dict.loc[lnc].tolist()
        mir_vec = mir_doc2vec_dict.loc[mir].tolist()
        mir_especifico_vec = mir_especifico_doc2vec_dict.loc[mir].tolist()
        H1, xedges, yedges = np.histogram2d(lnc_vec, mir_vec, bins=bin_size, density=True)
        H2, xedges, yedges = np.histogram2d(lnc_vec, mir_especifico_vec, bins=bin_size, density=True)
        H3, xedges, yedges = np.histogram2d(mir_especifico_vec, mir_vec, bins=bin_size, density=True)
        H = np.vstack((H1, H2, H3))
        H = minmax_scale(H)
        H_doc2vec = H[:, :, np.newaxis]

        lnc_vec = lnc_ctd_dict.loc[lnc].tolist()
        mir_vec = mir_ctd_dict.loc[mir].tolist()
        mir_especifico_vec = mir_especifico_ctd_dict.loc[mir].tolist()
        H1, xedges, yedges = np.histogram2d(lnc_vec, mir_vec, bins=bin_size, density=True)
        H2, xedges, yedges = np.histogram2d(lnc_vec, mir_especifico_vec, bins=bin_size, density=True)
        H3, xedges, yedges = np.histogram2d(mir_especifico_vec, mir_vec, bins=bin_size, density=True)
        H = np.vstack((H1, H2, H3))
        H = minmax_scale(H)
        H_ctd = H[:, :, np.newaxis]

        lnc_vec = lnc_role2vec_dict.loc[lnc].tolist()
        mir_vec = mir_role2vec_dict.loc[mir].tolist()
        mir_especifico_vec = mir_especifico_role2vec_dict.loc[mir].tolist()
        H1, xedges, yedges = np.histogram2d(lnc_vec, mir_vec, bins=bin_size, density=True)
        H2, xedges, yedges = np.histogram2d(lnc_vec, mir_especifico_vec, bins=bin_size, density=True)
        H3, xedges, yedges = np.histogram2d(mir_especifico_vec, mir_vec, bins=bin_size, density=True)
        H = np.vstack((H1, H2, H3))
        H = minmax_scale(H)
        H_role2vec = H[:, :, np.newaxis]

        feature_list = ss_loops_dict.loc[f'{lnc}_{mir}'].tolist()
        H = get_binding_feature(feature_list, bin_size)
        H_binding_feature_lnc = H[:, :, np.newaxis]

        feature_list = ss_mir_loops_dict.loc[f'{lnc}_{mir}'].tolist()
        H = get_binding_feature(feature_list, bin_size)
        H_binding_feature_mir = H[:, :, np.newaxis]

        mat = np.concatenate([H_kmer,H_doc2vec,H_ctd,H_role2vec, H_binding_feature_lnc, H_binding_feature_mir], axis=2)
        rows.append(mat)
            
test_data = np.array(rows)

100%|██████████| 45977/45977 [07:10<00:00, 106.74it/s]


In [5]:
gc.collect()

del rows
del data
del mat
del lnc_kmer_dict
del lnc_doc2vec_dict
del lnc_ctd_dict
del lnc_role2vec_dict

del mir_kmer_dict
del mir_doc2vec_dict
del mir_ctd_dict
del mir_role2vec_dict

del mir_especifico_kmer_dict
del mir_especifico_doc2vec_dict
del mir_especifico_ctd_dict
del mir_especifico_role2vec_dict

del ss_loops_dict
del ss_mir_loops_dict

del H1
del H2
del H

NameError: name 'data' is not defined

In [6]:
predicted = model.predict(test_data)
predicted

1437/1437 ━━━━━━━━━━━━━━━━━━━━ 665s 462ms/step


array([[2.37829071e-02, 9.76217031e-01],
       [1.96887888e-02, 9.80311215e-01],
       [1.32772857e-02, 9.86722708e-01],
       ...,
       [3.49728502e-02, 9.65027094e-01],
       [6.12996519e-02, 9.38700318e-01],
       [1.00000000e+00, 1.19692976e-08]], dtype=float32)

In [9]:
# Convertir 'predicted' en una lista de predicciones (probabilidades)
predicted_list = predicted.tolist()

# Crear una lista de filas para el CSV
csv_rows = []

# Iterar sobre los índices y las predicciones
for idx, pred in zip(indices, predicted_list):
    # Agregar el índice y la predicción a la lista de filas
    csv_rows.append([idx, pred[0], pred[1]])  # Si 'pred' es una lista de una sola probabilidad

# Crear un DataFrame de pandas con las filas del CSV
df = pd.DataFrame(csv_rows, columns=['Index', 'Probabilidad negative_pairs', 'Probabilidad positive_pairs'])

# Guardar el DataFrame en un archivo Excel
df.to_excel('./outputs/predictions.xlsx', index=False)

print("Archivo Excel generado y guardado como 'predictions.xlsx'")

Archivo Excel generado y guardado como 'predictions.xlsx'
